# Челленджи недели: функции, замыкания, декораторы, генераторы, with

**Цель:** проверить, что концепты недели работают вместе, а не каждый сам по себе.

**Как работать:**
- Часть задач сопровождается ячейкой «Объясни своими словами» — там нужно не только написать код, но и сформулировать, почему он работает.
- Все задачи решаются на 5-20 строк кода через `def`. Своих классов не пишем — `__enter__` / `__exit__` и пользовательские итераторы появятся на следующей неделе.
- Сначала попробуй сам, без подглядываний. Если застрял на 10+ минут — открой solution-версию.


## Задание 1: Декоратор-кэш для функции

Напиши декоратор `memoize`, который запоминает результаты вызовов функции в обычном словаре. При повторном вызове с теми же аргументами — возвращает запомненный результат, не пересчитывая.

Применишь его к рекурсивной `fib`. Без кэша `fib(35)` считается секунды; с кэшем — мгновенно.

Подсказки:
- кэш-словарь храни в замыкании
- ключом служит кортеж `args` (он хэшируемый)
- не забудь `@functools.wraps`

In [1]:
from functools import wraps

def memoize(func):
    cache = {}

    @wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]

    return wrapper

@memoize
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(35))    # 9227465 — мгновенно


9227465


**Объясни своими словами:** где живёт словарь `cache` после того, как `memoize` вернул `wrapper`, и почему он не сбрасывается между вызовами?

Словарь `cache` создан внутри `memoize` — это его локальная переменная. Когда `memoize` возвращает `wrapper`, `cache` обычно бы исчезла (локальные переменные функции живут только во время её вызова). Но `wrapper` использует `cache` — значит, ссылается на неё. Python видит эту ссылку и сохраняет переменную в замыкании. Каждый вызов `wrapper(...)` обращается к тому же объекту-словарю, и он накапливает результаты между вызовами.

## Задание 2: Генератор чисел Фибоначчи

Напиши генератор `fib_up_to(limit)`, который выдаёт числа Фибоначчи **меньше** `limit`. Последовательность начинается с `0, 1, 1, 2, 3, 5, 8, ...`.

Используй `yield` в `while`-цикле, без хранения всей последовательности в памяти.

Проверка: `list(fib_up_to(50))` должно вернуть `[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]`.

In [2]:
def fib_up_to(limit):
    a, b = 0, 1
    while a < limit:
        yield a
        a, b = b, a + b

print(list(fib_up_to(50)))    # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
print(list(fib_up_to(1)))     # [0]
print(list(fib_up_to(0)))     # []


[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
[0]
[]


## Задание 3: Контекстный менеджер для замера времени

Напиши контекстный менеджер `timed(label)` через `@contextlib.contextmanager`. Он замеряет, сколько прошло времени между входом и выходом из блока, и печатает результат.

Требования:
- замер работает даже при исключении внутри блока (используй `try/finally`)
- формат вывода: `[<label>] N.NNNN сек`

Проверь на `time.sleep(0.1)`.

In [3]:
from contextlib import contextmanager
import time

@contextmanager
def timed(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"[{label}] {elapsed:.4f} сек")

with timed("sleep"):
    time.sleep(0.1)

# Замер срабатывает и при исключении
try:
    with timed("при ошибке"):
        raise RuntimeError("упало")
except RuntimeError as e:
    print(f"поймали: {e}")


[sleep] 0.1049 сек
[при ошибке] 0.0000 сек
поймали: упало


**Объясни своими словами:** что произойдёт, если убрать `try/finally` и оставить только `yield` в теле менеджера?

Без `try/finally` код после `yield` (замер времени и печать) не выполнится при исключении внутри `with`-блока. Исключение «просочится» через `yield`, генератор завершится сразу, и до строк `elapsed = ... ; print(...)` дело не дойдёт. С `try/finally` блок `finally` гарантированно выполняется и при успехе, и при исключении — это тот же самый контракт, что и у `with` со встроенным менеджером файла.

## Задание 4: Подводный камень с изменяемым дефолтом

Дана «наивная» функция `add_item`, у которой дефолт-параметр `basket=[]`. Покажи, что несколько вызовов без явного `basket` делят один и тот же список (это и есть mutable default trap).

Затем напиши **исправленную** версию `add_item_safe`, которая ведёт себя ожидаемо: каждый вызов без `basket` получает свежий пустой список.

Стандартный шаблон починки — `basket=None` + проверка внутри функции.

In [4]:
def add_item(item, basket=[]):
    basket.append(item)
    return basket

# Демонстрация бага: дефолт делится между вызовами
print(add_item("яблоко"))    # ['яблоко']
print(add_item("груша"))     # ['яблоко', 'груша']  — bug
print(add_item("банан"))     # ['яблоко', 'груша', 'банан']  — bug

def add_item_safe(item, basket=None):
    if basket is None:
        basket = []
    basket.append(item)
    return basket

print(add_item_safe("яблоко"))    # ['яблоко']
print(add_item_safe("груша"))     # ['груша']  — каждый вызов свой список
print(add_item_safe("банан"))     # ['банан']


['яблоко']
['яблоко', 'груша']
['яблоко', 'груша', 'банан']
['яблоко']
['груша']
['банан']


**Объясни своими словами:** почему именно `None` как дефолт + проверка `if basket is None` — стандартный паттерн, а не, например, `basket=list()`?

Дефолт-выражение вычисляется **один раз** при объявлении функции, а не при каждом вызове. И `[]`, и `list()` создадут ровно один список в момент `def add_item(...)`, и все вызовы будут делить этот один список. `None` — неизменяемый объект, его можно безопасно использовать как «маркер пропущенного аргумента»: каждый вызов проверяет «если пользователь ничего не передал — создам новый список здесь, в теле функции». Тогда новый список рождается на каждом вызове.

## Задание 5: Фабрика счётчиков через замыкание

Напиши функцию-фабрику `make_counter(start=0, step=1)`, которая возвращает функцию-счётчик. Каждый вызов счётчика увеличивает его значение на `step` и возвращает текущее значение.

Состояние храни в замыкании. Помни: чтобы менять переменную из внешней области в Python, нужен `nonlocal`.

Проверка:
```
c = make_counter(start=10, step=5)
c()  # 15
c()  # 20
c()  # 25
```

In [5]:
def make_counter(start=0, step=1):
    count = start

    def increment():
        nonlocal count
        count += step
        return count

    return increment

c = make_counter(start=10, step=5)
print(c())    # 15
print(c())    # 20
print(c())    # 25

# Каждая фабрика создаёт независимый счётчик
c2 = make_counter()
print(c2())   # 1 — у c2 своя count
print(c2())   # 2
print(c())    # 30 — у c своя count


15
20
25
1
2
30


## Задание 6: Генераторное выражение для фильтрации логов

Дан список лог-строк. Напиши **одно** выражение, которое:

- оставляет только строки, начинающиеся с `ERROR`
- из каждой такой строки извлекает только сообщение (часть после `ERROR: `)

Используй генераторное выражение. Передай его в `list(...)`, чтобы материализовать результат.

In [6]:
log_lines = [
    "INFO: server started",
    "ERROR: db connection failed",
    "WARN: slow query",
    "ERROR: timeout after 30s",
    "INFO: request handled",
    "ERROR: invalid token",
]

errors = list(
    line[len("ERROR: "):]
    for line in log_lines
    if line.startswith("ERROR: ")
)

print(errors)
# ['db connection failed', 'timeout after 30s', 'invalid token']


['db connection failed', 'timeout after 30s', 'invalid token']


## Задание 7: Декоратор с аргументами — `retry`

Напиши декоратор `retry(times)`, который при падении функции с исключением — повторяет её до `times` раз. Если все попытки провалились — пробрасывает последнее исключение наружу.

Структура — три уровня вложенности: внешняя функция принимает `times`, возвращает декоратор, который возвращает `wrapper`.

Подсказка для проверки: внутри тестовой функции держим счётчик падений в замыкании и роняем её на первых двух вызовах, чтобы увидеть retry в действии.

In [7]:
from functools import wraps

def retry(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as exc:
                    last_exc = exc
                    print(f"попытка {attempt}/{times} упала: {exc}")
            raise last_exc
        return wrapper
    return decorator

# Тестовая функция, которая падает на первых двух вызовах
def make_flaky():
    calls = [0]
    def flaky():
        calls[0] += 1
        if calls[0] < 3:
            raise RuntimeError(f"вызов {calls[0]} провалился")
        return "ok"
    return flaky

@retry(times=3)
def task():
    return flaky()

flaky = make_flaky()
print(task())    # ok — третья попытка прошла


попытка 1/3 упала: вызов 1 провалился
попытка 2/3 упала: вызов 2 провалился
ok


**Объясни своими словами:** зачем нужен **третий** уровень вложенности (внешняя функция, декоратор, wrapper), и что бы случилось, если попробовать обойтись двумя?

Декоратор по контракту принимает только функцию (`func`) и возвращает функцию. Если мы хотим передать декоратору параметры (например, число повторов), нам нужна обёртка, которая принимает эти параметры и возвращает «обычный» декоратор. Получаются три уровня: внешняя `retry(times)` принимает аргумент → возвращает `decorator(func)` (это и есть «обычный» декоратор) → возвращает `wrapper(*args, **kwargs)` (это уже сама обёрнутая функция). Без третьего уровня `@retry(3)` не работает: `retry(3)` должна вернуть что-то, что ведёт себя как декоратор, а декоратор обязан принимать функцию.

## Задание 8: Сортировка словарей по нескольким полям

Дан список словарей с информацией о сотрудниках. Отсортируй его по двум полям одновременно: сначала по убыванию `salary`, затем (при равной зарплате) по возрастанию `name`.

Используй `sorted(...)` с параметром `key=` и `lambda`. Подсказка: вернуть из лямбды кортеж — `sorted` сравнит кортежи поэлементно. Чтобы отсортировать поле по убыванию — поставь перед ним знак `-` (для чисел) или используй параметр `reverse=` (сложнее, когда направления разные).

In [8]:
employees = [
    {"name": "Аня",   "salary": 100_000},
    {"name": "Боря",  "salary": 120_000},
    {"name": "Вера",  "salary": 100_000},
    {"name": "Гена",  "salary": 80_000},
    {"name": "Дима",  "salary": 120_000},
]

result = sorted(employees, key=lambda e: (-e["salary"], e["name"]))
for emp in result:
    print(emp)

# Ожидаемый порядок:
# Боря (120), Дима (120), Аня (100), Вера (100), Гена (80)


{'name': 'Боря', 'salary': 120000}
{'name': 'Дима', 'salary': 120000}
{'name': 'Аня', 'salary': 100000}
{'name': 'Вера', 'salary': 100000}
{'name': 'Гена', 'salary': 80000}


# Готово

Ты только что прошёл задачи на пересечении функций, замыканий, декораторов, генераторов и контекстных менеджеров. Если все ячейки прошли — концепты недели у тебя работают как единое целое.

На следующей неделе мы разберём ООП — и увидим, как замыкания и декораторы из этой недели превращаются в классы с состоянием, а пользовательские итераторы и контекстные менеджеры пишутся через магические методы.
